# Signal Diagnostics

IC time series, turnover, and summary table for all 10 signals.

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from src.data_bridge import load_all_signal_inputs
from src.signal_diagnostics import compute_ic_timeseries, ic_summary, compute_turnover

## Load signal panel

In [ ]:
panel = pd.read_parquet("../data/processed/signal_panel.parquet")
panel["date"] = pd.to_datetime(panel["date"])

sig_cols = [c for c in panel.columns if c not in ("ticker", "date")]
print(f"Panel shape: {panel.shape}")
print(f"Signals: {sig_cols}")
print(f"Date range: {panel['date'].min().date()} to {panel['date'].max().date()}")

## Build outcome: next-quarter stock return

Uses synthetic prices from `data_bridge` (same seed as the signal panel).
Outcome at each quarter-end = return from that quarter-end to the next quarter-end.
IC is computed at quarter-end dates so all signals (quarterly and monthly) align.

In [ ]:
data = load_all_signal_inputs(n_tickers=100, start="2012-01-01", end="2024-12-31")
prices = data["prices"]

wide = prices.pivot(index="date", columns="ticker", values="adj_close")
qprices = wide.resample("QE").last()

# Forward return: (price at next quarter-end) / (price now) - 1
fwd = qprices.shift(-1) / qprices - 1

outcome_df = (
    fwd.stack()
    .reset_index()
    .rename(columns={"ticker": "ticker", 0: "outcome"})
)
outcome_df.columns = ["date", "ticker", "outcome"]
outcome_df = outcome_df.dropna()

# Filter panel to quarter-end dates for consistent IC across all signals
quarter_ends = set(outcome_df["date"].unique())
panel_q = panel[panel["date"].isin(quarter_ends)].copy()

print(f"Outcome periods: {outcome_df['date'].nunique()} quarter-ends")
print(f"Panel rows at quarter-ends: {len(panel_q)}")

## IC time series and turnover per signal

In [ ]:
fig, axes = plt.subplots(5, 2, figsize=(14, 16), sharex=False)
axes = axes.flatten()

results = []

for i, col in enumerate(sig_cols):
    sig_df = (
        panel_q[["ticker", "date", col]]
        .dropna()
        .rename(columns={col: "signal"})
    )
    if len(sig_df) < 100:
        print(f"WARNING: {col} has too few observations ({len(sig_df)}), skipping.")
        axes[i].set_title(col)
        axes[i].text(0.5, 0.5, "insufficient data", ha="center", va="center",
                     transform=axes[i].transAxes)
        continue

    ic_ts = compute_ic_timeseries(sig_df, outcome_df)
    summary = ic_summary(ic_ts)
    turnover = compute_turnover(sig_df)

    results.append({
        "signal": col,
        "mean_ic": summary["mean_ic"],
        "std_ic": summary["std_ic"],
        "ir": summary["ir"],
        "pct_positive": summary["pct_positive"],
        "mean_turnover": turnover.mean(),
        "n_periods": summary["n_periods"],
    })

    print(f"{col:<12}  mean_ic={summary['mean_ic']:+.3f}  "
          f"std={summary['std_ic']:.3f}  IR={summary['ir']:+.2f}  "
          f"pct+={summary['pct_positive']:.0%}  "
          f"turnover={turnover.mean():.3f}")

    ax = axes[i]
    ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
    ax.bar(ic_ts.index, ic_ts.values, width=60, color=[
        "steelblue" if v >= 0 else "tomato" for v in ic_ts.values
    ], alpha=0.7)
    ax.plot(ic_ts.index, ic_ts.rolling(4).mean(), color="navy", linewidth=1.5,
            label="4-period MA")
    ax.set_title(f"{col}  (IR={summary['ir']:+.2f}, mean={summary['mean_ic']:+.3f})",
                 fontsize=9)
    ax.set_xlabel("")
    ax.tick_params(labelsize=7)
    ax.legend(fontsize=7)

# Hide unused subplots
for j in range(len(sig_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle("IC Time Series per Signal (vs next-quarter return)", fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig("../data/processed/ic_timeseries.png", bbox_inches="tight", dpi=100)
plt.show()
print("Saved IC chart to data/processed/ic_timeseries.png")

## Summary table

In [ ]:
summary_df = (
    pd.DataFrame(results)
    .sort_values("ir", ascending=False)
    .reset_index(drop=True)
)

# Flag signals with near-zero or negative mean IC
summary_df["flag"] = summary_df["mean_ic"].apply(
    lambda x: "WEAK" if abs(x) < 0.02 else ("NEG" if x < 0 else "")
)

fmt = {
    "mean_ic": "{:+.4f}",
    "std_ic": "{:.4f}",
    "ir": "{:+.3f}",
    "pct_positive": "{:.0%}",
    "mean_turnover": "{:.4f}",
}
print(summary_df.to_string(index=False, formatters={k: v.format for k, v in fmt.items()}))